# Capitolul 7: Construirea Aplicațiilor de Chat
## Începere rapidă cu API-ul OpenAI

Acest caiet este adaptat din [Depozitul de Mostre Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) care include caiete ce accesează servicii [Azure OpenAI](aoai-assignment.ipynb).

API-ul Python OpenAI funcționează și cu modelele Azure OpenAI, cu câteva modificări. Aflați mai multe despre diferențe aici: [Cum să comutați între punctele finale OpenAI și Azure OpenAI cu Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Prezentare generală  
"Modelele mari de limbaj sunt funcții care mapează text la text. Având un șir de text ca intrare, un model mare de limbaj încearcă să prezică textul care va urma"(1). Acest caiet de „pornire rapidă” va introduce utilizatorii în conceptele de nivel înalt ale LLM, cerințele de bază ale pachetului pentru a începe cu AML, o introducere ușoară în designul promptului și câteva exemple scurte de diferite cazuri de utilizare. 


## Cuprins  

[Prezentare generală](#overview)  
[Cum să utilizați serviciul OpenAI](#how-to-use-openai-service)  
[1. Crearea serviciului dvs. OpenAI](#1.-creating-your-openai-service)  
[2. Instalare](#2.-installation)    
[3. Credențiale](#3.-credentials)  

[Cazuri de utilizare](#use-cases)    
[1. Rezumarea textului](#1.-summarize-text)  
[2. Clasificarea textului](#2.-classify-text)  
[3. Generarea de noi nume de produse](#3.-generate-new-product-names)  
[4. Ajustarea fină a unui clasificator](#4.fine-tune-a-classifier)  

[Referințe](#references)


### Creează primul tău prompt  
Acest exercițiu scurt va oferi o introducere de bază pentru a trimite prompturi unui model OpenAI pentru o sarcină simplă de „sumarizare”.


**Pași**:  
1. Instalează biblioteca OpenAI în mediul tău Python  
2. Încarcă bibliotecile ajutătoare standard și setează credențialele tale de securitate OpenAI tipice pentru Serviciul OpenAI pe care l-ai creat  
3. Alege un model pentru sarcina ta  
4. Creează un prompt simplu pentru model  
5. Trimite cererea ta către API-ul modelului!


### 1. Instalează OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Importă librării ajutătoare și instanțiază acreditările


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Găsirea modelului potrivit  
Modele precum GPT-4o și GPT-4o mini pot înțelege și genera limbaj natural și sunt disponibile pe platforma OpenAI cu diferite niveluri de putere și viteză potrivite pentru diverse sarcini.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Proiectarea prompturilor  

"Magia modelelor mari de limbaj constă în faptul că, fiind antrenate să minimizeze această eroare de predicție pe cantități vaste de text, modelele ajung să învețe concepte utile pentru aceste predicții. De exemplu, ele învață concepte precum"(1):

* cum să scrii corect
* cum funcționează gramatica
* cum să parafrazezi
* cum să răspunzi la întrebări
* cum să ții o conversație
* cum să scrii în multe limbi
* cum să codezi
* etc.

#### Cum să controlezi un model mare de limbaj  
"Dintre toate intrările pentru un model mare de limbaj, de departe cea mai influentă este promptul text(1).

Modelele mari de limbaj pot fi promptate să producă ieșire în câteva moduri:

Instrucțiune: Spune modelului ce vrei
Completare: Induce modelul să completeze începutul a ceea ce vrei
Demonstrație: Arată modelului ce vrei, fie cu:
Câteva exemple în prompt
Sute sau mii de exemple într-un set de date de antrenament pentru fine-tuning"



#### Există trei linii directoare de bază pentru crearea prompturilor:

**Arată și spune**. Fă clar ce vrei fie prin instrucțiuni, exemple sau o combinație a celor două. Dacă vrei ca modelul să ordoneze o listă de elemente în ordine alfabetică sau să clasifice un paragraf după sentiment, arată-i că asta dorești.

**Furnizează date de calitate**. Dacă încerci să construiești un clasificator sau să faci modelul să urmeze un tipar, asigură-te că există suficiente exemple. Asigură-te că recitești exemplele — modelul este de obicei suficient de inteligent să vadă greșelile de ortografie de bază și să-ți ofere un răspuns, dar poate presupune și că acestea sunt intenționate și asta poate afecta răspunsul.

**Verifică setările.** Setările temperature și top_p controlează cât de determinist este modelul în generarea unui răspuns. Dacă ceri un răspuns unde există un singur răspuns corect, atunci ar trebui să le setezi mai jos. Dacă cauți răspunsuri mai diverse, atunci poate vrei să le setezi mai sus. Greșeala numărul unu pe care o fac oamenii cu aceste setări este să presupună că sunt controale pentru "inteligență" sau "creativitate".


Sursă: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Trimite!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Repetă aceeași apelare, cum se compară rezultatele? 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Rezumarea textului  
#### Provocare  
Rezumarea textului prin adăugarea unui 'tl;dr:' la finalul unui pasaj de text. Observați cum modelul înțelege să execute o serie de sarcini fără instrucțiuni suplimentare. Puteți experimenta cu prompturi mai descriptive decât tl;dr pentru a modifica comportamentul modelului și a personaliza rezumatul pe care îl primiți(3).  

Lucrările recente au demonstrat câștiguri substanțiale în multe sarcini și repere NLP prin pre-antrenarea pe un corpus mare de text urmată de ajustarea fină pe o sarcină specifică. Deși arhitectura este în mod tipic agnostică față de sarcină, această metodă încă necesită seturi de date pentru ajustare fină specifice sarcinii, cu mii sau zeci de mii de exemple. În contrast, oamenii pot, în general, să execute o sarcină nouă de limbaj doar din câteva exemple sau prin instrucțiuni simple – ceva ce sistemele NLP actuale încă se străduiesc să realizeze pe scară largă. Aici demonstrăm că scalarea modelelor de limbaj îmbunătățește semnificativ performanța agnostică față de sarcină, cu puține exemple, ajungând uneori chiar să rivalizeze cu abordările anterioare de ajustare fină, considerate de ultimă generație. 



Tl;dr


# Exerciții pentru mai multe cazuri de utilizare  
1. Rezumă Textul  
2. Clasifică Textul  
3. Generează Nume Noi de Produse


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Clasifică text  
#### Provocare  
Clasifică articolele în categorii oferite în momentul inferenței. În exemplul următor, oferim atât categoriile, cât și textul de clasificat în prompt(*playground_reference). 

Cerere client: Bună, una dintre tastele de pe tastatura laptopului meu s-a stricat recent și am nevoie de un înlocuitor:

Categorie clasificată:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Generează Nume Noi pentru Produse
#### Provocare
Creează nume de produse din cuvinte exemple. Aici includem în prompt informații despre produsul pentru care urmează să generăm nume. De asemenea, oferim un exemplu similar pentru a arăta tiparul pe care dorim să îl primim. Am setat și valoarea temperaturii ridicată pentru a crește aleatorietatea și răspunsurile mai inovative.

Descrierea produsului: Un aparat de făcut milkshake acasă
Cuvinte de pornire: rapid, sănătos, compact.
Nume de produse: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Descrierea produsului: O pereche de pantofi care se pot potrivi oricărei mărimi de picior.
Cuvinte de pornire: adaptabil, potrivit, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Referințe  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Cele mai bune practici pentru ajustarea fină a modelelor GPT pentru clasificarea textului](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Pentru Mai Mult Ajutor  
[Echipa de Comercializare OpenAI](AzureOpenAITeam@microsoft.com) 


# Contribuitori
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
